# Credit Card Approval Prediction - EDA & Modeling

This notebook demonstrates the complete end-to-end Machine Learning pipeline for predicting credit card approval. 

## Table of Contents:
1. Load Dataset
2. Exploratory Data Analysis (EDA)
3. Data Preprocessing
4. Model Training & Evaluation
5. Model Comparison & Selection

### 1. Load Dataset
We start by importing the necessary libraries and loading the dataset using Pandas.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from xgboost import XGBClassifier

# Set plotting aesthetic styles
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

In [ ]:
# Load the dataset
data_path = '../dataset/credit_card_data.csv'
df = pd.read_csv(data_path)
print(f"Dataset successfully loaded. Shape: {df.shape}")

### 2. Exploratory Data Analysis (EDA)
Let's look at the dataset properties, missing values, check feature distributions, and inspect relations using correlation analysis.

In [ ]:
# Display first 5 rows
df.head()

In [ ]:
# Dataset info
df.info()

In [ ]:
# Summary statistics for numerical columns
df.describe()

In [ ]:
# Summary statistics for categorical columns
df.describe(include=['object'])

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

#### Target Variable Distribution
Let's visualize the target column `Approved` (0 = Rejected, 1 = Approved) to see if we have an imbalanced dataset.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='Approved', data=df, palette='Set2')
plt.title('Target Variable (Approved) Distribution')
plt.xlabel('Approval Status (0 = Rejected, 1 = Approved)')
plt.ylabel('Count')
plt.show()

#### Feature Histograms
Let's plot histograms for numerical features to look at distributions.

In [ ]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
# Remove target variable from the histograms
if 'Approved' in num_cols:
    num_cols.remove('Approved')

fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    if i < len(axes):
        sns.histplot(df[col].dropna(), kde=True, ax=axes[i], color='skyblue')
        axes[i].set_title(f'Distribution of {col}')
        axes[i].set_xlabel('')
        axes[i].set_ylabel('')

# Hide unused axes
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

#### Correlation Heatmap
Let's check the linear correlation between the numerical columns.

In [ ]:
plt.figure(figsize=(10, 8))
# Select only numerical variables for correlation mapping
corr_matrix = df[num_cols + ['Approved']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, vmin=-1, vmax=1)
plt.title('Correlation Heatmap')
plt.show()

### 3. Data Preprocessing
We need to clean the data, impute missing values, scale numerical columns, and encode categorical columns before training the models.

In [ ]:
# Separate features (X) and target (y)
X = df.drop(columns=['Approved'])
y = df['Approved']

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
print(f"Numerical features to preprocess: {num_cols}")
print(f"Categorical features to preprocess: {cat_cols}")

In [ ]:
# Perform Train-Test Split (80:20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

In [ ]:
# Preprocessing pipelines for both types of features
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  # Impute missing values with median
    ('scaler', StandardScaler())                    # Scale values to have mean=0, std=1
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Impute missing categorical values with mode
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) # Encode categorical values
])

# ColumnTransformer combines the individual transformers
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

### 4. Model Training & Evaluation
We train and compare four classifiers:
1. Logistic Regression
2. Decision Tree Classifier
3. Random Forest Classifier
4. XGBoost Classifier

For each model, we will calculate accuracy, precision, recall, F1-score, and display the confusion matrix.

In [ ]:
# Define classifiers
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, eval_metric='logloss')
}

results = {}

for name, clf in classifiers.items():
    # Build a full pipeline combining preprocessing and classifier
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', clf)
    ])
    
    # Train the pipeline
    pipeline.fit(X_train, y_train)
    
    # Predict on test set
    y_pred = pipeline.predict(X_test)
    
    # Evaluate
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    
    results[name] = {
        'Pipeline': pipeline,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'Confusion Matrix': cm
    }
    
    # Print metrics
    print(f"\n{'='*15} {name} Evaluation {'='*15}")
    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-Score  : {f1:.4f}")
    print("Confusion Matrix:")
    print(cm)
    print(classification_report(y_test, y_pred))

### 5. Model Comparison & Selection
Let's tabulate and compare the results to identify the best performing classifier.

In [ ]:
comparison_data = []
for name, metrics in results.items():
    comparison_data.append({
        'Model': name,
        'Accuracy': metrics['Accuracy'],
        'Precision': metrics['Precision'],
        'Recall': metrics['Recall'],
        'F1-Score': metrics['F1-Score']
    })

df_compare = pd.DataFrame(comparison_data)
df_compare.sort_values(by='F1-Score', ascending=False, inplace=True)
df_compare

In [ ]:
# Plot comparison
df_melted = df_compare.melt(id_vars='Model', var_name='Metric', value_name='Score')
plt.figure(figsize=(12, 6))
sns.barplot(x='Model', y='Score', hue='Metric', data=df_melted, palette='viridis')
plt.title('Model Performance Comparison')
plt.ylim(0.7, 1.0)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

#### Save the Best Model
We serialize the pipeline of the best model (which includes the data preprocessor) to a `model.pkl` file, so it can be loaded directly in our Flask web application.

In [ ]:
best_model_name = df_compare.iloc[0]['Model']
best_pipeline = results[best_model_name]['Pipeline']
print(f"Best model is: {best_model_name}")

output_pkl = '../model.pkl'
with open(output_pkl, 'wb') as f:
    pickle.dump(best_pipeline, f)
print(f"Successfully saved the best model pipeline to {output_pkl}")